<a href="https://colab.research.google.com/github/Nahla-Nabil/agentic-distillation-benchmark/blob/master/notebooks/00_setup_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 — Colab setup

Run this first in every other notebook's runtime. Clones the repo, installs the GPU deps (`requirements-colab.txt`) plus this package in editable mode, and does a sanity check that `adbench` imports and a T4 is visible.

**Research question:** does a distilled student's multi-step agentic tool-use reliability degrade faster than its general LM performance, and at which layers does that gap originate?

In [2]:
# Public repo — no auth needed to clone. Skips re-cloning if this
# session's runtime already has the repo (e.g. you ran
# 00_setup_colab.ipynb earlier in this same session).
import os
import sys

# Reduces CUDA OOM from a single large allocation (e.g. loading the 14B
# teacher) by letting the allocator grow a segment incrementally instead of
# needing one big contiguous block upfront.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q -r requirements-colab.txt

# Put src/ on sys.path (for `import adbench` right here in this kernel)
# AND on PYTHONPATH (for `!python -m adbench...` subprocess calls in later
# cells, which inherit the environment but not this process's sys.path)
# instead of `pip install -e .` — an editable install registers itself via
# a .pth file that Python's site module only reads at interpreter startup,
# so `import adbench` fails with ModuleNotFoundError in this same
# still-running kernel until you restart it. Both of the below work
# immediately, no restart needed, on Colab or Kaggle.
src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

Cloning into '/kaggle/working/agentic-distillation-benchmark'...
remote: Enumerating objects: 276, done.
remote: Counting objects: 100% (276/276), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 276 (delta 143), reused 223 (delta 103), pack-reused 0 (from 0)
Receiving objects: 100% (276/276), 207.84 KiB | 6.93 MiB/s, done.
Resolving deltas: 100% (143/143), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 77.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [3]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
import adbench; print(adbench.__version__)

True Tesla T4
0.1.0
